# WR (Wide Receiver) Round Regression (Ridge)

Predict draft round 1–8 (8 = undrafted) for wide receivers using combine + RAS + PFF receiving metrics.

- **Train**: 2015–2023 from `wr_training.csv`
- **Test**: `wr_testing.csv` filtered to 2024/2025 (drafted only); 2026 predictions for all prospects.


In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

FEATURES_WITH_COLLEGE_WR = [
    # Physical / Athletic
    'Height', 'Weight', '40yd', 'Vertical', 'Broad Jump',
    'speed_score', 'explosive_score', 'RAS', 'arm_length_inches',
    # Receiving Efficiency / Core Production
    'yprr', 'yards_per_reception', 'caught_percent',
    'avg_depth_of_target', 'targeted_qb_rating',
    # Ball Skills / Hands
    'contested_catch_rate', 'drop_rate',
    # YAC / Open Field
    'yards_after_catch_per_reception', 'avoided_tackles',
    # Alignment / Usage
    'slot_rate', 'wide_rate', 'inline_rate', 'route_rate',
    # Experience / Context
    'player_game_count', 'p4_conference',
]

CONTAINS_WITH_COLLEGE_WR = [
    'contains_height', 'contains_weight', 'contains_40yd',
    'contains_vertical', 'contains_broad_jump',
    'contains_speed_score', 'contains_explosive_score',
    'contains_ras', 'contains_arm_length_inches',
    'contains_yprr', 'contains_yards_per_reception', 'contains_caught_percent',
    'contains_avg_depth_of_target', 'contains_targeted_qb_rating',
    'contains_contested_catch_rate', 'contains_drop_rate',
    'contains_yards_after_catch_per_reception', 'contains_avoided_tackles',
    'contains_slot_rate', 'contains_wide_rate', 'contains_inline_rate', 'contains_route_rate',
    'contains_player_game_count', 'contains_p4_conference',
]

FEATURES_ALL = FEATURES_WITH_COLLEGE_WR + CONTAINS_WITH_COLLEGE_WR

In [2]:
P4_PRE_2024 = {
    'Alabama', 'Arkansas', 'Auburn', 'Florida', 'Georgia', 'Kentucky',
    'LSU', 'Mississippi', 'Mississippi State', 'Missouri', 'South Carolina',
    'Tennessee', 'Texas A&M', 'Vanderbilt',
    'Illinois', 'Indiana', 'Iowa', 'Maryland', 'Michigan', 'Michigan State',
    'Minnesota', 'Nebraska', 'Northwestern', 'Ohio State', 'Penn State',
    'Purdue', 'Rutgers', 'Wisconsin',
    'Baylor', 'Iowa State', 'Kansas', 'Kansas State', 'Oklahoma',
    'Oklahoma State', 'TCU', 'Texas', 'Texas Tech', 'West Virginia',
    'Cincinnati', 'Houston', 'UCF', 'BYU',
    'Boston College', 'Clemson', 'Duke', 'Florida State', 'Georgia Tech',
    'Louisville', 'Miami', 'North Carolina', 'North Carolina State',
    'Pittsburgh', 'Syracuse', 'Virginia', 'Virginia Tech', 'Wake Forest',
    'Arizona', 'Arizona State', 'California', 'Colorado', 'Oregon',
    'Oregon State', 'Stanford', 'UCLA', 'USC', 'Utah', 'Washington', 'Washington State',
}
P4_2024_PLUS = (P4_PRE_2024
    - {'Arizona', 'Arizona State', 'California', 'Colorado', 'Oregon', 'Oregon State',
       'Stanford', 'UCLA', 'USC', 'Utah', 'Washington', 'Washington State'}
    | {'Oregon', 'Washington', 'UCLA', 'USC', 'Arizona', 'Arizona State', 'Utah', 'Colorado', 'SMU'}
)
P4_2024_PLUS -= {'Oregon State', 'Washington State', 'California', 'Stanford'}

def get_p4(school, year):
    ref = P4_PRE_2024 if year < 2024 else P4_2024_PLUS
    return 1 if str(school).strip() in ref else 0


def add_engineered_features(df):
    df = df.copy()
    # Height conversion
    if df['Height'].dtype == object or df['Height'].astype(str).str.contains('-', na=False).any():
        def _ht(h):
            if pd.isna(h): return np.nan
            s = str(h).strip()
            if '-' in s:
                p = s.split('-')
                try: return int(p[0]) * 12 + int(p[1])
                except: return np.nan
            try: return float(s)
            except: return np.nan
        df['Height'] = df['Height'].apply(_ht)
    else:
        df['Height'] = pd.to_numeric(df['Height'], errors='coerce')

    for c in ['Weight', '40yd', 'Vertical', 'Broad Jump', 'RAS', 'arm_length_inches',
              'yprr', 'yards_per_reception', 'caught_percent', 'avg_depth_of_target',
              'targeted_qb_rating', 'contested_catch_rate', 'drop_rate',
              'yards_after_catch_per_reception', 'avoided_tackles',
              'slot_rate', 'wide_rate', 'inline_rate', 'route_rate', 'player_game_count']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # speed_score
    w = df['Weight']; s = df['40yd']
    df['speed_score'] = np.where((s > 0) & (~w.isna()), w * 200 / (s ** 4), np.nan)

    # explosive_score: z-score of Vertical
    vert = df['Vertical']
    v_mean = vert.mean(); v_std = vert.std()
    df['explosive_score'] = np.where(~vert.isna(), (vert - v_mean) / (v_std + 1e-8), np.nan)

    # p4_conference
    df['p4_conference'] = df.apply(lambda r: get_p4(r['School'], int(r['Year'])), axis=1)

    # contains_* flags
    flag_map = {
        'contains_height': 'Height', 'contains_weight': 'Weight',
        'contains_40yd': '40yd', 'contains_vertical': 'Vertical',
        'contains_broad_jump': 'Broad Jump',
        'contains_speed_score': 'speed_score', 'contains_explosive_score': 'explosive_score',
        'contains_ras': 'RAS', 'contains_arm_length_inches': 'arm_length_inches',
        'contains_yprr': 'yprr', 'contains_yards_per_reception': 'yards_per_reception',
        'contains_caught_percent': 'caught_percent',
        'contains_avg_depth_of_target': 'avg_depth_of_target',
        'contains_targeted_qb_rating': 'targeted_qb_rating',
        'contains_contested_catch_rate': 'contested_catch_rate',
        'contains_drop_rate': 'drop_rate',
        'contains_yards_after_catch_per_reception': 'yards_after_catch_per_reception',
        'contains_avoided_tackles': 'avoided_tackles',
        'contains_slot_rate': 'slot_rate', 'contains_wide_rate': 'wide_rate',
        'contains_inline_rate': 'inline_rate', 'contains_route_rate': 'route_rate',
        'contains_player_game_count': 'player_game_count',
    }
    for flag, col in flag_map.items():
        df[flag] = df[col].notna().astype(int) if col in df.columns else 0
    df['contains_p4_conference'] = 1

    return df

In [3]:
# ── Load and prepare training data ──────────────────────────────────────────
df = pd.read_csv('../data/processed/wr_training.csv')
df = df[df['Year'].between(2015, 2023)].copy()
print(f'Train (2015–2023 WRs): {len(df)}')

df = add_engineered_features(df)
for c in FEATURES_ALL:
    if c not in df.columns:
        df[c] = 0

y = np.where(df['Drafted'].astype(bool), np.clip(df['Round'].fillna(1).astype(int), 1, 7), 8)
X_raw = df[FEATURES_ALL].copy()

imputer = KNNImputer(n_neighbors=10)
X = imputer.fit_transform(X_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_scaled, y)

y_pred_train = np.clip(ridge.predict(X_scaled), 1, 8)
print(f'Train MAE: {mean_absolute_error(y, y_pred_train):.4f}')

Train (2015–2023 WRs): 444
Train MAE: 1.4300


In [4]:
# ── Load testing data ─────────────────────────────────────────────────────
wr_testing = pd.read_csv('../data/processed/wr_testing.csv')

wr_2024 = wr_testing[(wr_testing['Year'] == 2024) & (pd.to_numeric(wr_testing['Round'], errors='coerce') < 8)].copy()
wr_2025 = wr_testing[(wr_testing['Year'] == 2025) & (pd.to_numeric(wr_testing['Round'], errors='coerce') < 8)].copy()
wr_2026 = wr_testing[wr_testing['Year'] == 2026].copy()

print(f'WR 2024 drafted: {len(wr_2024)}, 2025: {len(wr_2025)}, 2026 prospects: {len(wr_2026)}')


def prepare_wr_df(ldf, year):
    ldf = ldf.copy()
    ldf['Year'] = year
    ldf = add_engineered_features(ldf)
    for c in FEATURES_ALL:
        if c not in ldf.columns:
            ldf[c] = 0
    return ldf


def eval_metrics(actual, pred, label):
    mae  = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2   = r2_score(actual, pred)
    exact = (np.round(pred) == actual).mean()
    w1   = (np.abs(np.round(pred) - actual) <= 1).mean()
    print(f'{label} (n={len(actual)}): MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, '
          f'Exact={exact:.2%}, Within-1={w1:.2%}')

WR 2024 drafted: 35, 2025: 28, 2026 prospects: 46


In [5]:
# ── Evaluate 2024 and 2025 ────────────────────────────────────────────────
wr_2024 = prepare_wr_df(wr_2024, 2024)
wr_2025 = prepare_wr_df(wr_2025, 2025)

pred_24 = np.clip(ridge.predict(scaler.transform(imputer.transform(wr_2024[FEATURES_ALL]))), 1, 8)
pred_25 = np.clip(ridge.predict(scaler.transform(imputer.transform(wr_2025[FEATURES_ALL]))), 1, 8)

actual_24 = pd.to_numeric(wr_2024['Round'], errors='coerce').fillna(8).astype(int).values
actual_25 = pd.to_numeric(wr_2025['Round'], errors='coerce').fillna(8).astype(int).values

eval_metrics(actual_24, pred_24, '2024 WRs')
eval_metrics(actual_25, pred_25, '2025 WRs')

2024 WRs (n=35): MAE=1.4953, RMSE=1.9210, R²=0.2171, Exact=25.71%, Within-1=57.14%
2025 WRs (n=28): MAE=1.2843, RMSE=1.5505, R²=0.3444, Exact=21.43%, Within-1=60.71%


In [6]:
def wr_tier(p):
    if p < 1.75: return 'Round 1'
    if p < 2.75: return 'Round 2'
    if p < 3.75: return 'Round 3'
    if p < 4.75: return 'Round 4'
    if p < 5.75: return 'Round 5'
    if p < 6.75: return 'Round 6'
    if p < 7.75: return 'Round 7'
    return 'UDFA'

d24 = wr_2024[['Round', 'Pick', 'Player', 'School']].copy()
d24['predicted_round'] = pred_24; d24['tier'] = [wr_tier(x) for x in pred_24]
d24['Round'] = pd.to_numeric(d24['Round'], errors='coerce').astype('Int64')
print('2024 WRs')
display(d24.sort_values('predicted_round').reset_index(drop=True))

d25 = wr_2025[['Round', 'Pick', 'Player', 'School']].copy()
d25['predicted_round'] = pred_25; d25['tier'] = [wr_tier(x) for x in pred_25]
d25['Round'] = pd.to_numeric(d25['Round'], errors='coerce').astype('Int64')
print('2025 WRs')
display(d25.sort_values('predicted_round').reset_index(drop=True))

2024 WRs


,Round,Pick,Player,School,predicted_round,tier
0,1,6.00,Malik Nabers,LSU,1.00,Round 1
1,6,184.00,Malik Washington,Virginia,1.00,Round 1
2,2,34.00,Ladd McConkey,Georgia,2.51,Round 2
3,1,32.00,Xavier Legette,South Carolina,2.61,Round 2
4,4,102.00,Troy Franklin,Oregon,2.76,Round 3
5,1,28.00,Xavier Worthy,Texas,2.80,Round 3
6,1,9.00,Rome Odunze,Washington,2.87,Round 3
7,3,84.00,Roman Wilson,Michigan,3.19,Round 3
8,1,23.00,Brian Thomas Jr.,LSU,3.27,Round 3
9,3,80.00,Jermaine Burton,Alabama,3.84,Round 4


2025 WRs


,Round,Pick,Player,School,predicted_round,tier
0,2,55.00,Tre Harris,Mississippi,1.28,Round 1
1,4,108.00,Dont'e Thornton Jr.,Tennessee,2.28,Round 2
2,2,39.00,Luther Burden III,Missouri,2.54,Round 2
3,4,138.00,Jordan Watkins,Mississippi,2.78,Round 3
4,2,58.00,Jack Bech,TCU,3.24,Round 3
5,3,102.00,Tai Felton,Maryland,3.25,Round 3
6,5,158.00,KeAndre Lambert-Smith,Auburn,3.34,Round 3
7,4,133.00,Jalen Royals,Utah State,3.47,Round 3
8,1,19.00,Emeka Egbuka,Ohio State,3.47,Round 3
9,3,79.00,Jaylin Noel,Iowa State,3.61,Round 3


In [7]:
# ── 2026 WR predictions ───────────────────────────────────────────────────
wr_2026 = prepare_wr_df(wr_2026, 2026)
pred_26 = np.clip(ridge.predict(scaler.transform(imputer.transform(wr_2026[FEATURES_ALL]))), 1, 8)

d26 = wr_2026[['Player', 'School']].copy()
d26['Pos'] = 'WR'
d26['predicted_round'] = pred_26
d26['tier'] = [wr_tier(x) for x in pred_26]
d26_sorted = d26.sort_values('predicted_round').reset_index(drop=True)

print(f'2026 WR predictions (n={len(pred_26)})')
display(d26_sorted)

d26[['Player', 'School', 'Pos', 'predicted_round']].to_csv(
    '../data/processed/wr_2026_predictions.csv', index=False
)
print('Saved wr_2026_predictions.csv')

2026 WR predictions (n=46)


,Player,School,Pos,predicted_round,tier
0,Omar Cooper Jr,Indiana,WR,2.11,Round 2
1,Makai Lemon,USC,WR,2.66,Round 2
2,Skyler Bell,UConn,WR,2.83,Round 3
3,Denzel Boston,Washington,WR,2.88,Round 3
4,Zachariah Branch,Georgia,WR,3.02,Round 3
5,Jeff Caldwell,Cincinnati,WR,3.14,Round 3
6,De'Zhaun Stribling,Mississippi,WR,3.35,Round 3
7,KC Concepcion,Texas A&M,WR,3.54,Round 3
8,Carnell Tate,Ohio State,WR,3.58,Round 3
9,Brenen Thompson,Mississippi State,WR,3.70,Round 3


Saved wr_2026_predictions.csv


In [8]:
# ── Feature coefficients ─────────────────────────────────────────────────
coef_df = pd.DataFrame({'feature': FEATURES_ALL, 'coefficient': ridge.coef_})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)
print('Top 20 features by |coefficient|:')
display(coef_df.head(20).reset_index(drop=True))

Top 20 features by |coefficient|:


,feature,coefficient
0,slot_rate,1.10
1,wide_rate,1.09
2,yprr,-0.92
3,40yd,0.72
4,RAS,-0.55
5,caught_percent,0.47
6,targeted_qb_rating,-0.44
7,contains_speed_score,0.42
8,p4_conference,-0.40
9,yards_per_reception,0.37
